In [5]:
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

In [6]:
DATA_DIR = r"C:\Users\shrey\Training Practicals\archive(project_QM9)\dsgdb9nsd.xyz"
MAX_FILES = 500
RANDOM_STATE = 42

In [7]:
def load_qm9(data_dir, max_files):
    files = [f for f in os.listdir(data_dir) if f.endswith(".xyz")][:max_files]
    records = []

    for file in files:
        with open(os.path.join(data_dir, file)) as f:
            lines = f.readlines()

        try:
            num_atoms = int(lines[0].strip())
        except:
            continue

        values = []
        for token in lines[1].split():
            try:
                values.append(float(token))
            except:
                pass

        if len(values) < 5:
            continue

        energy = values[-5]  # U0 energy

        atom_counts = {"C":0, "H":0, "O":0, "N":0, "F":0}
        for line in lines[2:2 + num_atoms]:
            atom = line.split()[0]
            if atom in atom_counts:
                atom_counts[atom] += 1

        records.append({
            "num_atoms": num_atoms,
            **atom_counts,
            "energy": energy
        })

    return pd.DataFrame(records)

In [8]:
df = load_qm9(DATA_DIR, MAX_FILES)
df.head()

,num_atoms,C,H,O,N,F,energy
0,5,1,4,0,0,0,-40.478930
1,4,0,3,0,1,0,-56.525887
2,3,0,2,1,0,0,-76.404702
3,4,2,2,0,0,0,-77.308427
4,3,1,1,0,1,0,-93.411888


In [9]:
# Molecular weight
df["molecular_weight"] = (
    12.01 * df["C"] +
    1.008 * df["H"] +
    16.00 * df["O"] +
    14.01 * df["N"] +
    19.00 * df["F"]
)

# Heavy atom count
df["heavy_atoms"] = df["C"] + df["O"] + df["N"] + df["F"]

# Atom ratios (safe division)
df["H_C_ratio"] = df["H"] / df["C"].replace(0, np.nan)
df["O_C_ratio"] = df["O"] / df["C"].replace(0, np.nan)
df["N_C_ratio"] = df["N"] / df["C"].replace(0, np.nan)
df["F_C_ratio"] = df["F"] / df["C"].replace(0, np.nan)

df.fillna(0, inplace=True)

In [20]:
def to_formula(row):
    formula = ""
    for atom in ["C", "H", "O", "N", "F"]:
        count = int(row[atom])
        if count > 0:
            formula += atom if count == 1 else f"{atom}{count}"
    return formula

df["formula"] = df.apply(to_formula, axis=1)

In [21]:
FEATURES = [
    "num_atoms",
    "C", "H", "O", "N", "F",
    "heavy_atoms",
    "molecular_weight",
    "H_C_ratio", "O_C_ratio", "N_C_ratio", "F_C_ratio"
]

X = df[FEATURES]
y = df["energy"]

In [22]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

In [23]:
model = RandomForestRegressor(
    n_estimators=300,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

model.fit(X_train, y_train)

,n_estimators,300
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [24]:
y_pred = model.predict(X_test)

print("MAE:", mean_absolute_error(y_test, y_pred))
print("R² :", r2_score(y_test, y_pred))

MAE: 2.733128399596796
R² : 0.9844671163341091


In [25]:
importance_df = pd.DataFrame({
    "Feature": FEATURES,
    "Importance": model.feature_importances_
}).sort_values(by="Importance", ascending=False)

importance_df

,Feature,Importance
7,molecular_weight,0.732564
9,O_C_ratio,0.130282
1,C,0.036116
6,heavy_atoms,0.024493
3,O,0.018278
2,H,0.016801
0,num_atoms,0.009544
10,N_C_ratio,0.009005
8,H_C_ratio,0.007291
5,F,0.007171


In [26]:
def stability_level(energy):
    if energy < -200:
        return "High Stability"
    elif energy < -100:
        return "Medium Stability"
    else:
        return "Low Stability"

df["stability"] = df["energy"].apply(stability_level)

In [27]:
def energy_risk_score(energy):
    if energy > -80:
        return 3
    elif energy > -150:
        return 2
    else:
        return 1

def complexity_score(num_atoms):
    if num_atoms > 15:
        return 3
    elif num_atoms > 8:
        return 2
    else:
        return 1

def sustainability_score(energy, num_atoms):
    return (4 - energy_risk_score(energy)) + (4 - complexity_score(num_atoms))

df["energy_risk_score"] = df["energy"].apply(energy_risk_score)
df["complexity_score"] = df["num_atoms"].apply(complexity_score)

df["sustainability_score"] = df.apply(
    lambda x: sustainability_score(x["energy"], x["num_atoms"]),
    axis=1
)

In [28]:
df["energy_norm"] = (
    df["energy"] - df["energy"].min()
) / (df["energy"].max() - df["energy"].min())

df["complexity_norm"] = df["num_atoms"] / df["num_atoms"].max()
df["sustainability_norm"] = df["sustainability_score"] / df["sustainability_score"].max()

df["mo_score"] = (
    (1 - df["energy_norm"]) +
    (1 - df["complexity_norm"]) +
    df["sustainability_norm"]
)

In [29]:
def decision_readiness(score, stability):
    if score >= 6 and stability == "High Stability":
        return "Recommend"
    elif score >= 4:
        return "Investigate"
    else:
        return "Reject"

df["decision"] = df.apply(
    lambda x: decision_readiness(x["sustainability_score"], x["stability"]),
    axis=1
)

In [31]:
final_table = df[[
    # Identity
    "formula",
    
    # Core properties
    "energy",
    "num_atoms",
    "molecular_weight",
    "heavy_atoms",
    
    # Ratios
    "H_C_ratio", "O_C_ratio", "N_C_ratio", "F_C_ratio",
    
    # Scores
    "energy_risk_score",
    "complexity_score",
    "sustainability_score",
    "mo_score",
    
    # Labels
    "stability",
    "decision"
]].sort_values(
    by=["decision", "mo_score"],
    ascending=[True, False]
)

final_table.head(10)

,formula,energy,num_atoms,molecular_weight,heavy_atoms,H_C_ratio,O_C_ratio,N_C_ratio,F_C_ratio,energy_risk_score,complexity_score,sustainability_score,mo_score,stability,decision
24,C2N2,-185.648533,4,52.040,4,0.000000,0.000000,1.000000,0.0,1,1,6,2.165661,Medium Stability,Investigate
454,C3H4O3,-342.352279,10,88.062,6,1.333333,1.000000,0.000000,0.0,1,2,5,2.093708,High Stability,Investigate
451,C3H4O3,-342.335241,10,88.062,6,1.333333,1.000000,0.000000,0.0,1,2,5,2.093665,High Stability,Investigate
355,C3H4O3,-342.329232,10,88.062,6,1.333333,1.000000,0.000000,0.0,1,2,5,2.093650,High Stability,Investigate
430,C3H4O3,-342.315546,10,88.062,6,1.333333,1.000000,0.000000,0.0,1,2,5,2.093616,High Stability,Investigate
341,C3H4O3,-342.307696,10,88.062,6,1.333333,1.000000,0.000000,0.0,1,2,5,2.093596,High Stability,Investigate
356,C3H4O3,-342.297848,10,88.062,6,1.333333,1.000000,0.000000,0.0,1,2,5,2.093571,High Stability,Investigate
187,C3H3O2N,-321.276274,9,85.064,6,1.000000,0.666667,0.333333,0.0,1,2,5,2.090621,High Stability,Investigate
440,C3H3O2N,-321.271406,9,85.064,6,1.000000,0.666667,0.333333,0.0,1,2,5,2.090609,High Stability,Investigate
349,C3H3O2N,-321.254976,9,85.064,6,1.000000,0.666667,0.333333,0.0,1,2,5,2.090567,High Stability,Investigate


# test dataset

In [32]:
import pandas as pd

test_df = pd.DataFrame([
    # Simple small organic
    {
        "num_atoms": 6,
        "C": 2, "H": 4, "O": 0, "N": 0, "F": 0,
        "heavy_atoms": 2,
        "molecular_weight": 28.05,
        "H_C_ratio": 2.0, "O_C_ratio": 0.0,
        "N_C_ratio": 0.0, "F_C_ratio": 0.0
    },

    # Oxygen-containing molecule
    {
        "num_atoms": 9,
        "C": 3, "H": 6, "O": 1, "N": 0, "F": 0,
        "heavy_atoms": 4,
        "molecular_weight": 58.08,
        "H_C_ratio": 2.0, "O_C_ratio": 0.33,
        "N_C_ratio": 0.0, "F_C_ratio": 0.0
    },

    # Nitrogen-containing molecule
    {
        "num_atoms": 10,
        "C": 4, "H": 5, "O": 0, "N": 1, "F": 0,
        "heavy_atoms": 5,
        "molecular_weight": 67.07,
        "H_C_ratio": 1.25, "O_C_ratio": 0.0,
        "N_C_ratio": 0.25, "F_C_ratio": 0.0
    },

    # Fluorinated molecule
    {
        "num_atoms": 12,
        "C": 4, "H": 4, "O": 0, "N": 0, "F": 4,
        "heavy_atoms": 8,
        "molecular_weight": 100.04,
        "H_C_ratio": 1.0, "O_C_ratio": 0.0,
        "N_C_ratio": 0.0, "F_C_ratio": 1.0
    }
])

test_df

,num_atoms,C,H,O,N,F,heavy_atoms,molecular_weight,H_C_ratio,O_C_ratio,N_C_ratio,F_C_ratio
0,6,2,4,0,0,0,2,28.05,2.00,0.00,0.00,0.0
1,9,3,6,1,0,0,4,58.08,2.00,0.33,0.00,0.0
2,10,4,5,0,1,0,5,67.07,1.25,0.00,0.25,0.0
3,12,4,4,0,0,4,8,100.04,1.00,0.00,0.00,1.0


In [33]:
predicted_energy = model.predict(test_df)

In [34]:
predicted_energy

array([ -95.11420225, -193.60373057, -210.19476053, -298.89464228])